In [91]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

from scipy import signal
from neuralhydrology.datautils import utils
from neuralhydrology.evaluation.metrics import _validate_inputs, _mask_valid

In [92]:
# ------------------- Paths -------------------
RUN_DIR = Path("../runs")

ensemble_metrics_dir=Path("./ensemble_peak_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [93]:
# run_pattern = "precip_prcp_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"


matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/test/model_epoch030/test_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2904_123501', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_222_2904_124105', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_333_2904_124709', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_444_2904_125314', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_555_2904_125917', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_666_2904_130523', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_777_2904_131125', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_888_2904_131729']


In [94]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['QObs_mm_d_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['QObs_mm_d_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'CAMELS_UY_10': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:        (date: 2191, time_step: 1)
   Coordinates:
     * date           (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
     * time_step      (time_step) int64 8B 0
   Data variables:
       QObs_mm_d_obs  (date, time_step) float32 9kB 0.2376 0.2216 ... 1.265 1.686
       QObs_mm_d_sim  (date, time_step) float32 9kB 0.2122 0.2132 ... 0.5629 0.5599}},
 'CAMELS_UY_11': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:        (date: 2191, time_step: 1)
   Coordinates:
     * date           (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
     * time_step      (time_step) int64 8B 0
   Data variables:
       QObs_mm_d_obs  (date, time_step) float32 9kB 0.3589 0.3589 ... 0.8041 0.6947
       QObs_mm_d_sim  (date, time_step) float32 9kB 0.4802 0.4038 ... 0.8334 0.6516}},
 'CAMELS_UY_15': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:        (date: 2191, time_step: 1)
   Coordinates:
     * dat

In [95]:
def custom_mean_peak_timing(obs, sim, window=None, resolution='1D', datetime_coord=None, distance=100):
    _validate_inputs(obs, sim)
    obs, sim = _mask_valid(obs, sim)

    peaks, _ = signal.find_peaks(obs.values, distance=distance, prominence=np.std(obs.values))

    if datetime_coord is None:
        datetime_coord = utils.infer_datetime_coord(obs)
    if window is None:
        window = max(int(utils.get_frequency_factor('12h', resolution)), 3)

    timing_errors = []
    for idx in peaks:
        if (idx - window < 0) or (idx + window >= len(obs)) or (
            pd.date_range(obs[idx - window][datetime_coord].values,
                          obs[idx + window][datetime_coord].values,
                          freq=resolution).size != 2 * window + 1):
            continue

        if (sim[idx] > sim[idx - 1]) and (sim[idx] > sim[idx + 1]):
            peak_sim = sim[idx]
        else:
            values = sim[idx - window:idx + window + 1]
            peak_sim = values[values.argmax()]

        peak_obs = obs[idx]
        delta = peak_obs.coords[datetime_coord] - peak_sim.coords[datetime_coord]
        timing_errors.append(np.abs(delta.values / pd.to_timedelta(resolution)))

    return np.mean(timing_errors) if len(timing_errors) > 0 else np.nan


def custom_missed_peaks(obs, sim, window=None, resolution='1D', percentile=80, datetime_coord=None, distance=30):
    _validate_inputs(obs, sim)
    obs, sim = _mask_valid(obs, sim)

    min_obs_height = np.percentile(obs.values, percentile)
    min_sim_height = np.percentile(sim.values, percentile)

    peaks_obs_times, _ = signal.find_peaks(obs, distance=distance, height=min_obs_height)
    peaks_sim_times, _ = signal.find_peaks(sim, distance=distance, height=min_sim_height)

    if len(peaks_obs_times) == 0:
        return 0.

    if datetime_coord is None:
        datetime_coord = utils.infer_datetime_coord(obs)
    if window is None:
        window = max(int(utils.get_frequency_factor('12h', resolution)), 1)

    missed_events = 0
    for idx in peaks_obs_times:
        if (idx - window < 0) or (idx + window >= len(obs)) or (
            pd.date_range(obs[idx - window][datetime_coord].values,
                          obs[idx + window][datetime_coord].values,
                          freq=resolution).size != 2 * window + 1):
            continue

        nearby_peak_sim_index = np.where(np.abs(peaks_sim_times - idx) <= window)[0]
        if len(nearby_peak_sim_index) == 0:
            missed_events += 1

    return missed_events / len(peaks_obs_times)

In [96]:
PEAK_TIMING_DISTANCE = 100
MISSED_PEAKS_DISTANCE = 100
WINDOW = 5

In [97]:
all_metrics = {}
for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    obs = xr_ds['QObs_mm_d_obs']
    sim = xr_ds['QObs_mm_d_sim']
    
    if obs.isnull().all() or sim.isnull().all():
        print(f"Skipping {basin_id} — all observed/simulated values are NaN")
        continue
    
    all_metrics[basin_id] = {
        'Peak-Timing': custom_mean_peak_timing(obs, sim, window=WINDOW, distance=PEAK_TIMING_DISTANCE, resolution="1D", datetime_coord="date"),
        'Missed-Peaks': custom_missed_peaks(obs, sim, window=WINDOW, distance=MISSED_PEAKS_DISTANCE, resolution="1D", datetime_coord="date")
    }

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'
df_metrics

Skipping CAMELS_UY_15 — all observed/simulated values are NaN


,Peak-Timing,Missed-Peaks
basin_id,,
CAMELS_UY_10,1.833333,0.250000
CAMELS_UY_11,1.200000,0.583333
CAMELS_UY_16,0.250000,0.428571
CAMELS_UY_2,2.222222,0.181818
CAMELS_UY_3,0.545455,0.454545
CAMELS_UY_5,1.000000,0.538462
CAMELS_UY_6,1.636364,0.500000
CAMELS_UY_7,0.750000,0.500000
CAMELS_UY_8,1.333333,0.250000


In [98]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(ensemble_metrics_dir/f"{save_name}.csv")

In [99]:
df_metrics.median()

Peak-Timing     1.266667
Missed-Peaks    0.441558
dtype: float64